## Geneformer Fine-Tuning for Cell Annotation Application

In [1]:
import os
GPU_NUMBER = [0]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join([str(s) for s in GPU_NUMBER])
os.environ["NCCL_DEBUG"] = "INFO"
os.environ["WANDB_DISABLED"] = "true"

In [2]:
# imports
from collections import Counter
import datetime
import pickle
import subprocess
import seaborn as sns; sns.set()
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import BertConfig, BertForSequenceClassification
from transformers import Trainer
from transformers.training_args import TraianingArguments

from geneformer import DataCollatorForCellClassification
import sys
import re
import numpy as np
import torch

/opt/venv/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/opt/venv/lib/python3.8/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://git

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/token/dictionary/MLM-re_token_dictionary_v1.pkl'

## Prepare training and evaluation datasets

In [ ]:
# load cell type or disease dataset (includes all tissues)


# select fine turning type (ctc: cell type classification or isp: in silico perturbation)
f_type = "ctc"

# dataset_name(xxx.dataset path)
dataset_name = "/work/eval_dataset/all_organ_normal_mouse_tokenize_easy_dataset_v-n1.dataset"

# load dataset
train_dataset = load_from_disk(dataset_name)

# check and remove column names
if f_type == "isp" :
    try :
        print(np.unique(train_dataset["disease"]))
    except KeyError as e :
        print("KeyError: {}".format(e))
        print("changing to disease")
        train_dataset = train_dataset.rename_column("column name in diseases infomation","disease")
        print("change finished")
        print(np.unique(train_dataset["disease"]))
    
elif f_type == "ctc" :
    try :
        print(np.unique(train_dataset["cell_type"]))
    except KeyError as e :
        print("KeyError: {}".format(e))
        print("changing to cell_type")
        train_dataset = train_dataset.rename_column("column name in cell types infomation","cell_type")
        print("change finished")
        print(np.unique(train_dataset["cell_type"]))

else :
    print("error: select fine turning type (ctc or isp)")
    sys.exit(1)


print(train_dataset)


In [ ]:
# remove cache files in xxx.dataset 
 
import glob
import os
import tqdm
from tqdm.notebook import tqdm
from tqdm import tqdm_notebook as tqdm

rmfiles = glob.glob(dataset_name+"/cache*")

if rmfiles == [] :
    print("not exist cache files")
else :
    for tqdm_i2, rmfile in zip(tqdm(rmfiles, desc='remove files loop'), rmfiles) :
        os.remove(rmfile)
    print("Finished removeing cache file in it !!")

## Cell Type Classification

In [ ]:
# If you do cell type classification task, you have to run this cell. Otherwise, if you do disease type classification (in silico perturbation) task, you have to run the cell below.

dataset_list = []
evalset_list = []
organ_list = []
target_dict_list = []


for organ in Counter(train_dataset["organ_major"]).keys():
    # collect list of tissues for fine-tuning (immune and bone marrow are included together)
    if organ in ["bone_marrow"]:  
        continue
    elif organ=="immune":
        organ_ids = ["immune","bone_marrow"]
        organ_list += ["immune"]
    else:
        organ_ids = [organ]
        organ_list += [organ]
    
    print(organ)
    
    # filter datasets for given organ
    def if_organ(example):
        return example["organ_major"] in organ_ids
    trainset_organ = train_dataset.filter(if_organ, num_proc=16)
    
    # per scDeepsort published method, drop cell types representing <0.5% of cells
    celltype_counter = Counter(trainset_organ["cell_type"])
    total_cells = sum(celltype_counter.values())
    cells_to_keep = [k for k,v in celltype_counter.items() if v>(0.005*total_cells)]
    def if_not_rare_celltype(example):
        return example["cell_type"] in cells_to_keep
    trainset_organ_subset = trainset_organ.filter(if_not_rare_celltype, num_proc=16)
      
    # shuffle datasets and rename columns
    trainset_organ_shuffled = trainset_organ_subset.shuffle(seed=42)
    trainset_organ_shuffled = trainset_organ_shuffled.rename_column("cell_type","label")
    trainset_organ_shuffled = trainset_organ_shuffled.remove_columns("organ_major")
    
    # create dictionary of cell types : label ids
    target_names = list(Counter(trainset_organ_shuffled["label"]).keys())
    target_name_id_dict = dict(zip(target_names,[i for i in range(len(target_names))]))
    target_dict_list += [target_name_id_dict]
    
    # change labels to numerical ids
    def classes_to_ids(example):
        example["label"] = target_name_id_dict[example["label"]]
        return example
    labeled_trainset = trainset_organ_shuffled.map(classes_to_ids, num_proc=16)
    
    # create 80/20 train/eval splits
    labeled_train_split = labeled_trainset.select([i for i in range(0,round(len(labeled_trainset)*0.8))])
    labeled_eval_split = labeled_trainset.select([i for i in range(round(len(labeled_trainset)*0.8),len(labeled_trainset))])
    
    # filter dataset for cell types in corresponding training set
    trained_labels = list(Counter(labeled_train_split["label"]).keys())
    def if_trained_label(example):
        return example["label"] in trained_labels
    labeled_eval_split_subset = labeled_eval_split.filter(if_trained_label, num_proc=16)

    dataset_list += [labeled_train_split]
    evalset_list += [labeled_eval_split_subset]

## Disease Type Classification

In [ ]:
# disease classification

dataset_list = []
evalset_list = []
organ_list = []
target_dict_list = []


for organ in Counter(train_dataset["organ_major"]).keys():
    # collect list of tissues for fine-tuning (immune and bone marrow are included together)
    
    organ_ids = [organ]
    organ_list += [organ]
    
    print(organ)

    # filter datasets for given organ
    def if_organ(example):
        return example["organ_major"] in organ_ids
    trainset_organ = train_dataset.filter(if_organ, num_proc=16)
    
    # per scDeepsort published method, drop cell types representing <0.5% of cells
    celltype_counter = Counter(trainset_organ["disease"])
    total_cells = sum(celltype_counter.values())
    cells_to_keep = [k for k,v in celltype_counter.items() if v>(0.005*total_cells)]
    def if_not_rare_celltype(example):
        return example["disease"] in cells_to_keep
    trainset_organ_subset = trainset_organ.filter(if_not_rare_celltype, num_proc=16)
      
    # shuffle datasets and rename columns
    trainset_organ_shuffled = trainset_organ_subset.shuffle(seed=42)
    trainset_organ_shuffled = trainset_organ_shuffled.rename_column("disease","label")
    trainset_organ_shuffled = trainset_organ_shuffled.remove_columns("organ_major")
    
    # create dictionary of cell types : label ids
    target_names = list(Counter(trainset_organ_shuffled["label"]).keys())
    target_name_id_dict = dict(zip(target_names,[i for i in range(len(target_names))]))
    target_dict_list += [target_name_id_dict]
    
    # change labels to numerical ids
    def classes_to_ids(example):
        example["label"] = target_name_id_dict[example["label"]]
        return example
    labeled_trainset = trainset_organ_shuffled.map(classes_to_ids, num_proc=16)
    
    # create 80/20 train/eval splits
    labeled_train_split = labeled_trainset.select([i for i in range(0,round(len(labeled_trainset)*0.8))])
    labeled_eval_split = labeled_trainset.select([i for i in range(round(len(labeled_trainset)*0.8),len(labeled_trainset))])
    
    # filter dataset for cell types in corresponding training set
    trained_labels = list(Counter(labeled_train_split["label"]).keys())
    def if_trained_label(example):
        return example["label"] in trained_labels
    labeled_eval_split_subset = labeled_eval_split.filter(if_trained_label, num_proc=16)

    dataset_list += [labeled_train_split]
    evalset_list += [labeled_eval_split_subset]

In [ ]:
# this cell must run

trainset_dict = dict(zip(organ_list,dataset_list))
traintargetdict_dict = dict(zip(organ_list,target_dict_list))

evalset_dict = dict(zip(organ_list,evalset_list))


print(trainset_dict)
print(traintargetdict_dict)

print(evalset_dict)


## Fine-Tune With Cell Classification Learning Objective and Quantify Predictive Performance

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    acc = accuracy_score(labels, preds)
    pre = precision_score(labels, preds, average='macro')
    rec = recall_score(labels, preds, average='macro')
    macro_f1 = f1_score(labels, preds, average='macro')
    return {
      'accuracy': acc,
      'macro_precision': pre,
      'macro_recall': rec,
      'macro_f1': macro_f1
    }

### Please note that, as usual with deep learning models, we **highly** recommend tuning learning hyperparameters for all fine-tuning applications as this can significantly improve model performance. Example hyperparameters are defined below, but please see the "hyperparam_optimiz_for_disease_classifier" script for an example of how to tune hyperparameters for downstream applications.

In [ ]:
# set model parameters
# max input size
max_input_size = 2**11  # 2048

# set training hyperparameters
# max learning rate
max_lr = 5e-5
# how many pretrained layers to freeze
freeze_layers = 0
# number gpus
num_gpus = 1
# number cpu cores
num_proc = 16
# batch size for training and eval
geneformer_batch_size = 6
# learning schedule
lr_schedule_fn = "linear" #"polynomial", "linear", "cosine"
# warmup steps
warmup_steps = 500
# number of epochs
epochs = 10
# optimizer
optimizer = "adamW"


In [ ]:
import os
import datetime
import subprocess
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from transformers import (
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from geneformer.collator_for_classification import DataCollatorForCellClassification

import umap
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# -----------------------------------------------------------------------------
# 設定
# -----------------------------------------------------------------------------
warnings.filterwarnings(
    "ignore",
    message="To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach",
    category=UserWarning
)

do_umap   = False    # 最終 run のみ UMAP を出力するか
num_runs  = 1       # 必要に応じて複数 run に
base_seed = 42

# -----------------------------------------------------------------------------
# 実験ループ
# -----------------------------------------------------------------------------
for organ in organ_list:
    print(f"{organ} を処理中．")
    train_ds   = trainset_dict[organ]
    eval_ds    = evalset_dict[organ]
    label_dict = traintargetdict_dict[organ]
    print(f"ラベル辞書: {label_dict}")
    # 反転してインデックス→名前マップを作成
    inv_label_dict = {v: k for k, v in label_dict.items()}

    accuracy_list, f1_list = [], []
    best_acc, best_run, best_df = -np.inf, None, None

    for run_idx in range(num_runs):
        seed = base_seed + run_idx
        print(f"\n--- Run {run_idx+1}/{num_runs} (seed={seed}) ---")
        set_seed(seed)

        # モデルロード (hidden_states=False)
        checkpoint = "/work/mouse-Geneformer++/runs/20251023/training/checkpoint-2984040"
        model = BertForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=len(label_dict),
            output_attentions=False,
            output_hidden_states=False,
            ignore_mismatched_sizes=True
        ).to("cuda")

        # BERT の最大系列長 & pad_id
        max_len = model.config.max_position_embeddings
        pad_id  = model.config.pad_token_id or 0

        # 切り詰め＆パディング関数
        def _truncate_and_pad(batch):
            new_ids, new_masks, new_lens = [], [], []
            has_mask = "attention_mask" in batch
            for idx, ids in enumerate(batch["input_ids"]):
                # 切り詰め
                trunc = ids[:max_len]
                length = min(batch.get("length",[len(ids)])[idx], max_len)
                # pad input_ids
                if len(trunc) < max_len:
                    trunc = trunc + [pad_id] * (max_len - len(trunc))
                new_ids.append(trunc)
                # attention_mask
                if has_mask:
                    mask = batch["attention_mask"][idx][:max_len]
                    if len(mask) < max_len:
                        mask = mask + [0]*(max_len-len(mask))
                else:
                    mask = [1]*length + [0]*(max_len-length)
                new_masks.append(mask)
                new_lens.append(length)
            batch["input_ids"]      = new_ids
            batch["attention_mask"] = new_masks
            batch["length"]         = new_lens
            return batch

        train_ds = train_ds.map(_truncate_and_pad, batched=True)
        eval_ds  = eval_ds.map(_truncate_and_pad,  batched=True)

        train_ds.set_format("torch", columns=["input_ids","attention_mask","label","length"])
        eval_ds.set_format( "torch", columns=["input_ids","attention_mask","label","length"])

        # 出力ディレクトリ準備
        now    = datetime.datetime.now()
        date   = f"{now.year%100:02d}{now.month:02d}{now.day:02d}"
        out_dir = f"result/{date}_{organ}_run{run_idx+1}/"
        os.makedirs(out_dir, exist_ok=True)

        # TrainingArguments
        log_steps = max(1, len(train_ds)//geneformer_batch_size//10)
        args = TrainingArguments(
            output_dir=out_dir,
            learning_rate = max_lr,
            seed          = seed,
            do_train      = True,
            do_eval       = True,
            eval_strategy = "epoch",
            save_strategy = "epoch",
            logging_steps = log_steps,
            fp16          = True,
            dataloader_num_workers = num_proc,
            dataloader_pin_memory  = True,
            group_by_length        = True,
            length_column_name     = "length",
            disable_tqdm           = False,
            lr_scheduler_type      = lr_schedule_fn,
            warmup_steps           = warmup_steps,
            weight_decay           = 0.001,
            per_device_train_batch_size = geneformer_batch_size,
            per_device_eval_batch_size  = max(1, geneformer_batch_size//4),
            num_train_epochs            = epochs,
            load_best_model_at_end      = True,
            eval_accumulation_steps     = 1,
        )

        trainer = Trainer(
            model          = model,
            args           = args,
            data_collator = DataCollatorForCellClassification(),
            train_dataset  = train_ds,
            eval_dataset   = eval_ds,
            compute_metrics=compute_metrics,
        )

        # train & eval
        trainer.train()
        torch.cuda.empty_cache()
        preds = trainer.predict(eval_ds)
        torch.cuda.empty_cache()

        # metrics
        acc = preds.metrics.get("test_accuracy")
        f1  = preds.metrics.get("test_macro_precision")
        if acc is not None: accuracy_list.append(acc)
        if f1  is not None: f1_list.append(f1)

        # best 更新
        if acc is not None and acc > best_acc:
            best_acc, best_run = acc, run_idx+1
            pred_ids = np.argmax(preds.predictions, axis=1)
            best_df  = pd.DataFrame({
                "True_Label": [
                    inv_label_dict.get(int(i),"Unknown") for i in preds.label_ids
                ],
                "Pred_Label": [
                    inv_label_dict.get(int(i),"Unknown") for i in pred_ids
                ],
                "Logits": [l.tolist() for l in preds.predictions]
            })

        print(f"  acc={acc*100:.1f}％ (best run={best_run})")

        # 保存
        trainer.save_model(out_dir)
        trainer.save_metrics("eval", preds.metrics)
        with open(os.path.join(out_dir,"preds.pkl"),"wb") as fp:
            pickle.dump(preds, fp)
        pd.DataFrame({
            "True": [
                inv_label_dict.get(int(i),"Unknown") for i in preds.label_ids
            ],
            "Pred": [
                inv_label_dict.get(int(i),"Unknown") for i in np.argmax(preds.predictions,axis=1)
            ]
        }).to_csv(os.path.join(out_dir,"results.csv"), index=False)

        # --- UMAP (最終 run のみ) ---
        if do_umap and run_idx == num_runs-1:
            del model; torch.cuda.empty_cache()
            umap_model = BertForSequenceClassification.from_pretrained(
                out_dir, output_hidden_states=True
            ).to("cuda")
            umap_model.eval()

            loader, embs, lbls = DataLoader(
                eval_ds,
                batch_size=max(1, geneformer_batch_size//2),
                collate_fn=trainer.data_collator
            ), [], []
            with torch.no_grad():
                for b in loader:
                    inp = {k:v.to("cuda") for k,v in b.items()
                           if k in ["input_ids","attention_mask"]}
                    out = umap_model(**inp)
                    hs  = out.hidden_states[-1]
                    cls = hs[:,0,:].half().cpu().numpy()
                    embs.append(cls)
                    lbls.extend(b["labels"].cpu().numpy())

            del umap_model, out, hs, inp; torch.cuda.empty_cache()
            embs = np.vstack(embs).astype(np.float32)
            lbls = np.array(lbls)

            # PCA→UMAP
            n_samp, n_feat = embs.shape
            comp = min(50, n_samp, n_feat)
            red  = PCA(n_components=comp, random_state=seed).fit_transform(embs)
            proj = umap.UMAP(
                n_components=2,
                random_state=seed,
                low_memory=True,
                n_neighbors=15
            ).fit_transform(red)

            # プロット＋凡例（名前付き）
            plt.figure(figsize=(8,8))
            scatter = plt.scatter(
                proj[:,0], proj[:,1],
                c=lbls, cmap="tab10",
                s=10, alpha=0.7
            )
            unique = np.unique(lbls)
            handles = []
            for u in unique:
                handles.append(
                    plt.Line2D([], [], marker="o", linestyle="",
                               color=scatter.cmap(scatter.norm(u)),
                               label=inv_label_dict.get(int(u), str(u)))
                )
            plt.legend(handles=handles, title="Condition", loc="best")
            plt.title(f"{organ} の特徴空間 UMAP")
            plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")

            pdf_path = os.path.join(out_dir, "umap_projection_no_y.pdf")
            plt.savefig(pdf_path, format="pdf", dpi=300, bbox_inches="tight")
            plt.close()
            print(f"UMAP プロットを保存: {pdf_path}．")

    # 統計表示・ベスト結果保存
    if accuracy_list:
        m, s = np.mean(accuracy_list)*100, np.std(accuracy_list)*100
        print(f"\n=== {organ} 精度 ===\n平均: {m:.1f}％，標準偏差: {s:.1f}％")
    if best_df is not None:
        best_dir = os.path.join(f"result/{date}_Best_{organ}/")
        os.makedirs(best_dir, exist_ok=True)
        best_df.to_csv(os.path.join(best_dir,"best_results.csv"), index=False)
        print(f"ベスト結果 CSV を保存: {best_dir}．")


In [11]:
# remove cache files in xxx.dataset

rmfiles = glob.glob(dataset_name+"/cache*")
#print(rmfiles)
for tqdm_i2, rmfile in zip(tqdm(rmfiles, desc='remove files loop'), rmfiles) :
    os.remove(rmfile)
print("Finished removing cache files in this dataset!!")


/tmp/ipykernel_142421/448546242.py:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for tqdm_i2, rmfile in zip(tqdm(rmfiles, desc='remove files loop'), rmfiles) :


remove files loop:   0%|          | 0/603 [00:00<?, ?it/s]

Finished removing cache files in this dataset!!


#### import os

output_dir = "result/250407_mouse-geneformer_CellClassifier_Cop1KO_L4096_B12_LR5e-05_LSlinear_WU500_E20_OadamW_F0_ISP-Cop1KO"
print("pytorch_model.bin exists:", os.path.exists(os.path.join(output_dir, "pytorch_model.bin")))

# ディレクトリ内のファイル一覧を表示
print(os.listdir(output_dir) if os.path.exists(output_dir) else "Directory does not exist")


In [12]:
from transformers import AutoConfig

# safetensors から config を取得
checkpoint_path = "result/250407_mouse-geneformer_CellClassifier_Cop1KO_L4096_B12_LR5e-05_LSlinear_WU500_E20_OadamW_F0_ISP-Cop1KO"

config = AutoConfig.from_pretrained(checkpoint_path)
config.save_pretrained(checkpoint_path)  # config.json を保存



In [13]:
import os
print(os.path.exists(os.path.join(checkpoint_path, "config.json")))


True


In [14]:
import shutil

# 変換後の .bin ファイルのパス
checkpoint_path = "result/250407_mouse-geneformer_CellClassifier_Cop1KO_L4096_B12_LR5e-05_LSlinear_WU500_E20_OadamW_F0_ISP-Cop1KO"
bin_file_path = os.path.join(checkpoint_path, "config.json")

# 保存したいローカルフォルダを指定（例: ~/models/）
local_save_path = os.path.expanduser("pretrain")  # Linux/Mac のホームディレクトリに保存
os.makedirs(local_save_path, exist_ok=True)  # フォルダがなければ作成

# ファイルをコピー
shutil.copy(bin_file_path, os.path.join(local_save_path, "config.json"))

print(f"pytorch_model.bin copied to {local_save_path}")


pytorch_model.bin copied to pretrain
